# 5. The quantum support vector machine

The other half of section 17: replace the classical kernel with a quantum
one.


In [ ]:
import math
import sys
from pathlib import Path

# Make the shared qc101 library importable, whether the notebook is opened
# from the repository root or from inside the notebooks/ directory.
_here = Path.cwd().resolve()
for _candidate in (_here / "labs" / "common",
                   _here.parent / "labs" / "common",
                   _here / ".." / "labs" / "common"):
    if (_candidate / "qc101").is_dir():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise SystemExit("Could not find the qc101 library -- run this notebook "
                     "from the repository root or from notebooks/.")

from qc101 import boolean, complexnum, linalg, photon, probability, quantum, viz
from qc101.labkit import Lab

print("qc101 library ready")


In [ ]:
import numpy as np

## The classical kernel and the quantum one

$$K(x, z) = \phi(x) \cdot \phi(z)
\qquad\longrightarrow\qquad
K(x, z) = |\langle\phi(x)|\phi(z)\rangle|^2$$


In [ ]:
try:
    from qiskit.circuit.library import zz_feature_map
    from qiskit.quantum_info import Statevector

    feature_map = zz_feature_map(feature_dimension=4, reps=2, entanglement="linear")
    print(feature_map.decompose())
except ImportError as exc:
    print("Qiskit is not installed:", exc)

## A kernel entry, computed from the state vectors

The kernel is just the squared overlap of two feature-map states.


In [ ]:
try:
    def feature_state(x):
        return Statevector(feature_map.assign_parameters(list(x)))

    x1 = [5.0, 3.3, 1.5, 0.3]
    x2 = [6.0, 2.9, 5.2, 1.7]
    k12 = abs(np.vdot(feature_state(x1).data, feature_state(x2).data)) ** 2
    k11 = abs(np.vdot(feature_state(x1).data, feature_state(x1).data)) ** 2
    print(f"K(x1, x1) = {k11:.6f}")
    print(f"K(x1, x2) = {k12:.6f}")
except ImportError as exc:
    print("Qiskit is not installed:", exc)

## Training a QSVC

The course's `qsvm.py`, modernized: `BasicAer` and `QuantumInstance` were
removed in Qiskit 1.0.


In [ ]:
try:
    from sklearn.datasets import load_iris
    from qiskit_machine_learning.algorithms import QSVC
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    from qiskit_machine_learning.primitives import QMLSampler
    from qiskit_machine_learning.state_fidelities import ComputeUncompute

    iris = load_iris()
    kernel = FidelityQuantumKernel(
        feature_map=feature_map,
        fidelity=ComputeUncompute(sampler=QMLSampler()),
    )
    qsvc = QSVC(quantum_kernel=kernel)
    qsvc.fit(iris.data, iris.target)
    print("predictions:",
          qsvc.predict([[5.0, 3.3, 1.5, 0.3], [6.0, 2.9, 5.2, 1.7]]))
except ImportError as exc:
    print("qiskit-machine-learning is not installed:", exc)

## What this is actually worth

Quantum machine learning is mainly a new way to compute a *similarity*.
The data wrangling, the loss and the optimizer stay classical.
